# From drp_stella's guiders to drp_qa

Each call of `pfs.drp.stella.utils.guiders` beside its replacement in `pfs.drp.qa.guiders`, run on the same visits,
so that you can switch a notebook over by finding its calls here. What each plot shows and how to read it is in
[`guider-plots.ipynb`](guider-plots.ipynb).

It needs a real opdb, both packages (drp_stella's `guiders` is removed in PIPE2D-1901), and pfs_utils 7.4.18 or
later. The visits are engineering visits of Run 30; change them to yours.

What changes everywhere:

- **Read once, then analyse and plot.** Only `queries` touches the opdb or a butler. Analysis and plots take the
  DataFrame `readAgcData` returns, never a connection, and don't modify it.
- **Fit, then plot.** drp_stella's plots fitted and plotted in one call; now the fit (`analysis`) returns its result,
  and the plot (`plotting`) draws it.
- **Hardware coordinates, center minus reference.** Positions are flipped to hardware coordinates as they are read;
  every offset is a measured center minus a reference, in microns. `showAgcErrorsForVisits`'s x and y change sign.
- **Units in names, cameras 0-5.** `maxGuideError` is `maxGuideError_um`, `rms` is `rms_pix`; cameras are
  `agcCameraIds` 0-5 (AG1 is 0) rather than `AGC` 1-6.
- **Valid matches only.** Fits and averages use only matches with `agc_match_flags` GOOD_MATCH; `readAgcData`
  returns every match, so the others can still be looked at.
- **Axes, not pyplot.** Plots draw on the `ax`, `axes` or `fig` they are given (or a new figure) and return a
  `GuiderPlot`; `livePlot` becomes `axes=` and `colorbars=`.

| drp_stella `guiders.` | drp_qa |
|---|---|
| `readAgcDataFromOpdb(opdb, visits, butler)` | `queries.readAgcData(opdb, visits, butler=butler)` |
| `readAGCPositionsForVisitByAgcExposureId(opdb, visit, True)` | `queries.readAgcData(opdb, visit)` |
| `readAGCStarsForVisitSetByPfsVisitId(opdb, visits)` | `analysis.addImageSizes(queries.readAgcData(opdb, visits))` |
| `readAGCStars`, `readPfsDesign`, `readSpSInfo`, `readTelStatus`, `find_W_M2OFF3` | the same names in `queries` |
| `GuiderConfig(...)`, `showGuiderErrors(agcData, config)` | `fit = analysis.fitGuiderModel(agcData, analysis.GuiderFitConfig(...))`, `plotting.showGuiderErrors(fit, plotting.GuiderPlotConfig(...))` |
| `showGuiderErrorsByParams(agcData, guideErrorByCamera, params, config)` | `plotting.showGuiderErrorsByParams(fit, params)` |
| `showAgcErrorsForVisits`, `showAgcErrorsForVisitsByCamera`, `showTelescopeErrors` | the same names in `plotting`, given AG data |
| `plotDriftRate(agcData, rates=rates)` | `plotting.plotDriftRate(analysis.fitDriftRate(agcData))` |
| `estimateGuideErrors(agcData, plot=True)` | `plotting.plotGuideErrors(analysis.estimateGuideErrors(agcData))` |
| `compareAGCPfsUtils(opdb, visit)` | `plotting.plotPfsUtilsComparison(analysis.comparePfsUtilsPositions(agcStars, agcData))` |
| `plotFocus(opdb, visits)`, `plotFocusByAG(opdb, visits)` | `plotting.plotFocus(agcData)`, `plotting.plotFocusByAG(agcData)` |
| `MomentDifferenceToMmPiston`, `setImageSizes`, `averageArraysByFocusPosition`, `smoothAgcData` | `analysis.momentDifferenceToPiston`, `analysis.addImageSizes`, `analysis.averageByFocusPosition`, `analysis.smoothAgcData` |
| `set_center0`, `set_nominal0` | `coordinates.addReferencePositions(agcData, "center0")`, `"nominal0"` |
| `rotXY`, `agcCameraCenters`, `get_guiderFocus_to_M2_OFF3()` | `coordinates.rotXY`, `coordinates.AGC_CAMERA_CENTERS_MM`, `coordinates.guiderFocusToM2Off3` |
| `ag_to_zenith_offset.ag_pfimm_to_zenith_offset`, `fit_global_model_pfimm` | `coordinates.pfiToZenith`, `analysis.fitGlobalModel` |

## Setup

Give the notebook the opdb. drp_qa's readers take a `pfs.utils.database.opdb.OpDB`; drp_stella's take a psycopg2
connection, made here from the same parameters. Use a read-only user. A Gen3 butler is only needed for INST-PA and,
before 2025-03-21, M2_OFF3.

In [ ]:
%matplotlib inline

import warnings

import matplotlib.pyplot as plt
import psycopg2

from pfs.drp.qa.guiders import analysis, compat, coordinates, plotting, queries
from pfs.drp.stella.utils import guiders
from pfs.utils.database.opdb import OpDB

plt.rcParams["figure.figsize"] = (10, 7)

opdb = OpDB(host="pfsa-db", user="public_user")  # drp_qa
legacyOpdb = psycopg2.connect(opdb.dsn)  # drp_stella
butler = None

VISIT = 148258  # an all-sky exposure
VISITS = list(range(148284, 148292))  # a raster scan
FOCUS_VISITS = [148266, *range(148270, 148278)]  # a focus sweep

## Reading AG data

`readAgcData` is one query for all the visits, returns one row per matched spot (every match, with its `agc_match_flags`), in hardware coordinates, with a fresh index and the columns of `queries.AGC_DATA_COLUMNS`. It reads INST-PA through a Gen3 butler; drp_stella's asked a Gen2 butler for `raw_md`, which fails. drp_stella's averaged the spots matched to one guide star in an AG exposure into one row, with the smallest of their flags, so a valid and an invalid match could be averaged together; drp_qa's has a row for each.

**drp_stella**

In [ ]:
legacyAgcData = guiders.readAgcDataFromOpdb(legacyOpdb, VISITS)

**drp_qa**

In [ ]:
agcData = queries.readAgcData(opdb, VISITS, butler=butler)

In [ ]:
print(
    f"drp_stella: {len(legacyAgcData)} rows; drp_qa: {len(agcData)} rows, {analysis.selectValidMatches(agcData).mean():.0%} valid"
)
print("Only in drp_stella's:", sorted(set(legacyAgcData.columns) - set(agcData.columns)))
print("Only in drp_qa's:", sorted(set(agcData.columns) - set(legacyAgcData.columns)))

Until a notebook is moved over, `compat` keeps drp_stella's reader names, with their arguments and columns, on top of
`readAgcData`; each raises a `DeprecationWarning`.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    compatAgcData = compat.readAgcDataFromOpdb(opdb, VISITS)
compatAgcData.head()

## Reading the stars, with their sizes

drp_stella's star readers added `rms`, `FWHM` and `left`; `addImageSizes` adds `rms_pix`, `fwhm_arcsec` and `left` to any AG data. `useTraceRadius` is the same option.

**drp_stella**

In [ ]:
legacyStars = guiders.readAGCStarsForVisitSetByPfsVisitId(legacyOpdb, FOCUS_VISITS)

**drp_qa**

In [ ]:
focusData = queries.readAgcData(opdb, FOCUS_VISITS, butler=butler)
stars = analysis.addImageSizes(focusData)

## A design's guide stars, and the other readers

Same names and arguments. `readAGCStars` has drp_stella's missing comma fixed (`guide_star_parallax` came back as `ra_center_design`), and the schema's `*_designed` column names. drp_stella's fails on today's opdb, where `guide_star_ra` is in two of the tables it joins; so does its `compareAGCPfsUtils`, below.

**drp_stella**

In [ ]:
legacyDesign = guiders.readPfsDesign(legacyOpdb, VISIT)
legacyTelStatus = guiders.readTelStatus(legacyOpdb, VISIT)
try:
    legacyAgcStars = guiders.readAGCStars(legacyOpdb, legacyDesign.pfs_design_id.iloc[0], VISIT)
except Exception as error:  # guide_star_ra is ambiguous
    legacyOpdb.rollback()
    print(f"drp_stella's readAGCStars fails: {type(error).__name__}")

**drp_qa**

In [ ]:
design = queries.readPfsDesign(opdb, VISIT)
agcStars = queries.readAGCStars(opdb, design.pfs_design_id.iloc[0], VISIT)
telStatus = queries.readTelStatus(opdb, VISIT)  # or a list of visits

In [ ]:
allSky = queries.readAgcData(opdb, VISIT, butler=butler)
legacyAllSky = guiders.readAgcDataFromOpdb(legacyOpdb, [VISIT])

## Guide error of each AG exposure

The x and y errors change sign: drp_stella plotted nominal minus center. `pfs_visit_ids`, `agc_exposure_ids`, `yminmax`, `figure` and `livePlot` are `pfsVisitIds`, `agcExposureIds`, `yLimit_um`, `fig` and `axes`.

**drp_stella**

In [ ]:
guiders.showAgcErrorsForVisits(legacyAgcData)

**drp_qa**

In [ ]:
plot = plotting.showAgcErrorsForVisits(agcData)

## Each camera's guide errors

Give it AG data rather than the opdb and visits. `AGC=[1, 2]` is `agcCameraIds=[0, 1]`; `xminmax` and `yminmax` are `xLimit_um` and `yLimit_um`; `showFocusSets`, which did nothing, is gone.

**drp_stella**

In [ ]:
legacyStarsRaster = guiders.showAgcErrorsForVisitsByCamera(legacyOpdb, visits=VISITS, plotPerCamera=True)

**drp_qa**

In [ ]:
plot = plotting.showAgcErrorsForVisitsByCamera(agcData, plotPerCamera=True)

## The guide stars on the PFI

`GuiderConfig` held the fit's and the plot's options and cached the fitted transforms; now the fit's options are `analysis.GuiderFitConfig`, the plot's `plotting.GuiderPlotConfig`, and the transforms are in the fit (`fit.exposureParameters`, `fit.cameraParameters`; pass `previous=fit` to reuse them). Renamed: `maxGuideError` → `maxGuideError_um`, `maxPosError` → `maxPosError_um`, `agc_exposure_idsStride` → `agcExposureStride`, `pfs_visitIdMin` → `pfsVisitIdMin`, `agc_exposure_idMin` → `agcExposureIdMin`, `guideErrorEstimate` → `guideErrorEstimate_um`, `guide_star_frac` → `guideStarFrac`, `agc_exposure_cm` → `colormap`, `showByVisitSize` → `markerSize`, `showByVisitAlpha` → `alpha`. These are ics_pfsPlotActor's settings. drp_stella's modified its input, hence the copy. On this visit drp_stella's panel is empty: its guide error counts the invalid matches (29% of them, hundreds of microns off), so every AG exposure fails the 100 µm cut.

**drp_stella**

In [ ]:
legacyConfig = guiders.GuiderConfig(
    transforms={},
    modelBoresightOffset=False,
    modelCCDOffset=False,
    maxGuideError=100,
    maxPosError=40,
    guide_star_frac=0.3,
)
plt.figure()
legacyGuideErrorByCamera = guiders.showGuiderErrors(legacyAllSky.copy(), legacyConfig)

**drp_qa**

In [ ]:
fitConfig = analysis.GuiderFitConfig(
    modelBoresightOffset=False, modelCCDOffset=False, maxGuideError_um=100, maxPosError_um=40
)
plotConfig = plotting.GuiderPlotConfig(guideStarFrac=0.3)
fit = analysis.fitGuiderModel(allSky, fitConfig)
plot = plotting.showGuiderErrors(fit, plotConfig)
fit.guideErrorByCamera.head()

## Coloured by other quantities

The fit carries what drp_stella passed separately. drp_stella's fails here, as its `showGuiderErrors` averaged no stars on this visit.

**drp_stella**

In [ ]:
try:
    guiders.showGuiderErrorsByParams(
        legacyAllSky, legacyGuideErrorByCamera, ["altitude", "insrot"], legacyConfig
    )
except IndexError:  # drp_stella's showGuiderErrors, above, averaged no stars on this visit
    print("drp_stella's showGuiderErrorsByParams fails: no stars")

**drp_qa**

In [ ]:
plot = plotting.showGuiderErrorsByParams(fit, ["altitude", "insrot"], plotConfig)

## The guide corrections sent to the telescope

No config. `showTheta=True` now plots the rotator correction in arcsec (drp_stella's drew an empty panel).

**drp_stella**

In [ ]:
guiders.showTelescopeErrors(legacyAllSky, legacyConfig)

**drp_qa**

In [ ]:
plot = plotting.showTelescopeErrors(allSky)

## Drift

Fit, then plot. The rates are in `drift.rates` rather than a `rates` dict (whose default was shared between calls, and whose x rate was overwritten by the y rate). `guideStrategy` is `reference`; `showNominal=True` is `position="nominal"`; `agcVisitSmoothing` is `smoothing`; `visitName` is `name`.

**drp_stella**

In [ ]:
rates = {}
guiders.plotDriftRate(legacyAllSky.copy(), rates=rates)
rates

**drp_qa**

In [ ]:
drift = analysis.fitDriftRate(allSky)
plot = plotting.plotDriftRate(drift)
drift.rates

## Each camera's guide errors on the PFI

drp_stella's `estimateGuideErrors(plot=True)` is the analysis `estimateGuideErrors` and the plot `plotGuideErrors`. `guideStrategy` is `reference` (`center0PerVisit` → `center0_visit`, `nominal0PerVisit` → `nominal0_visit`), `xy0Stat` is `stat`, `agcVisitSmoothing` is `smoothing`, `showClosedShutter` is `includeClosedShutter` (and `showClosedShutter` in the plot); `byVisit`/`labelByTime` colour by `colorBy`. The result's `xbar`, `ybar` (mm) are `dx_um`, `dy_um` (µm).

**drp_stella**

In [ ]:
plt.figure()
_, legacyGuideErrors = guiders.estimateGuideErrors(legacyAllSky, plot=True)

**drp_qa**

In [ ]:
guideErrors = analysis.estimateGuideErrors(allSky)
plot = plotting.plotGuideErrors(guideErrors)

## The guider's and pfs_utils's positions

Give it the guide stars and the AG data. pfs_utils's positions now come from the AG actor's own model and inputs, and agree with the guider's to well under a micron; drp_stella's used the cobras' model, about 550 µm off, and gave pfs_utils HST as UTC. `instPaCorrection` is `instPaCorrection_arcsec`; `delta_theta` is `delta_theta_arcsec`, of the opposite sign (hardware coordinates).

**drp_stella**

In [ ]:
plt.figure()
try:
    legacyComparison = guiders.compareAGCPfsUtils(legacyOpdb, VISIT)
except Exception as error:  # its readAGCStars: guide_star_ra is ambiguous
    legacyOpdb.rollback()
    print(f"drp_stella's compareAGCPfsUtils fails: {type(error).__name__}")

**drp_qa**

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # pfs_utils's clamped parallaxes
    comparison = analysis.comparePfsUtilsPositions(agcStars, allSky)
plot = plotting.plotPfsUtilsComparison(comparison)

## Focus

Give it AG data rather than the opdb and visits (and no butler: the reader takes it). `AGC` is `agcCameraIds` (0-5); `yLimitsMicron`, `minFWHM` and `maxFWHM` are `yLimits_um`, `minFWHM_arcsec` and `maxFWHM_arcsec`; `mmToMicrons` is gone (always µm). `plotBy="agc_exposure_id"`, `showFocusSets` and clicking to set M2_OFF3 now work.

**drp_stella**

In [ ]:
legacyFocusStars = guiders.plotFocus(legacyOpdb, FOCUS_VISITS)

**drp_qa**

In [ ]:
plot = plotting.plotFocus(focusData, fig=plt.figure(figsize=(10, 10)))

## Each camera's focus relative to the others

Give it AG data. `minAgcExposureId`, `maxAgcExposureId` and `maxFocusError` are `agcExposureIdMin`, `agcExposureIdMax` and `maxFocusError_um`; `onlyGuideStars` now selects the isolated GAIA stars.

**drp_stella**

In [ ]:
plt.figure()
guiders.plotFocusByAG(legacyOpdb, FOCUS_VISITS)

**drp_qa**

In [ ]:
plot = plotting.plotFocusByAG(focusData)

## Helpers

Renamed, with their units; `smoothAgcData` now follows each star and leaves IDs and flags alone.

**drp_stella**

In [ ]:
from pfs.drp.stella.utils import ag_to_zenith_offset

legacyPiston = guiders.MomentDifferenceToMmPiston(0.5, includePistonCorrection=True)
legacySmoothed = guiders.smoothAgcData(legacyAllSky, 5)
legacyCenter0 = guiders.set_center0(legacyAllSky)
legacyZenith = ag_to_zenith_offset.ag_pfimm_to_zenith_offset(
    legacyAllSky.agc_nominal_x_mm, -legacyAllSky.agc_nominal_y_mm, legacyAllSky.insrot
)
guiders.agcCameraCenters, guiders.get_guiderFocus_to_M2_OFF3()

**drp_qa**

In [ ]:
piston = analysis.momentDifferenceToPiston(0.5, includePistonCorrection=True)
smoothed = analysis.smoothAgcData(allSky, 5)
center0 = coordinates.addReferencePositions(allSky, "center0", stat="mean")  # agc_center0_[xy]_mm
zenith = coordinates.pfiToZenith(
    allSky.agc_nominal_x_mm, allSky.agc_nominal_y_mm, allSky.insrot
)  # hardware y
coordinates.AGC_CAMERA_CENTERS_MM, coordinates.guiderFocusToM2Off3(1.0)

## In ics_pfsPlotActor

Its panels called drp_stella's plots with `livePlot`, or its own figure and axes; drp_qa's take its axes, and the colorbars of the previous
draw, which they update rather than add again. Its `AxesGrid` workaround for `plotFocus` is no longer needed.

In [ ]:
fig = plt.figure(figsize=(16, 8))
grid = fig.add_gridspec(3, 2)
left = [fig.add_subplot(grid[i, 0]) for i in range(3)]
right = fig.add_subplot(grid[:, 1])

plotting.showAgcErrorsForVisits(allSky, axes=left)
guider = plotting.showGuiderErrors(fit, plotConfig, ax=right)
# On the next update: clear the axes, then
# plotting.showGuiderErrors(fit, plotConfig, ax=right, colorbars=guider.colorbars)